# Module 5: Gateway & Identity -- Connecting External APIs

![Overview](../shared/img/05.drawio.png)

In this module you will give Aria the ability to **manage tasks** by connecting her to an external REST API through AgentCore Gateway, with JWT-based identity so every user's data stays private.

---

### What you will learn

| Topic | Details |
|---|---|
| **AgentCore Gateway** | An MCP endpoint that routes agent tool requests to backend APIs |
| **MCP protocol** | How Gateway auto-discovers API structure and exposes it as MCP tools |
| **Identity & JWT auth** | How `CUSTOM_JWT` auth validates user tokens against Cognito |
| **End-to-end identity flow** | User JWT flows from Runtime through Gateway to the target API |

## Catch-up

The cell below ensures all resources from previous modules (Runtime, Memory) are in place.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared.ensure_ready import ensure_ready

config = ensure_ready("05")

---

## Understanding Gateway

AgentCore Gateway is a **managed MCP endpoint** that sits between your agent and backend APIs. Instead of writing custom tool code for every API, you point Gateway at a backend service and it:

1. **Auto-discovers** the API structure (paths, methods, parameters)
2. **Generates MCP tools** that your agent can call (e.g., `list_tasks`, `create_task`)
3. **Handles authentication** between the agent and the target API
4. **Routes requests** to the target using IAM credentials from the Gateway's execution role

### Target types

Gateway supports five types of backend targets:

| Target Type | Description |
|---|---|
| **Lambda** | Invoke a Lambda function directly |
| **API Gateway** | Auto-discover an API Gateway REST API (what we use in this module) |
| **OpenAPI** | Provide an OpenAPI spec for any HTTP endpoint |
| **Smithy** | Provide a Smithy model definition |
| **MCP server** | Connect to another MCP-compatible server |

### Authentication modes

Gateway supports multiple authentication modes for validating inbound requests:

| Auth Mode | Use Case |
|---|---|
| `NONE` | Public APIs or development/testing |
| `CUSTOM_JWT` | User-facing APIs where each request carries a JWT identifying the user |
| `AWS_IAM` | AWS service-to-service authentication |

In this module we use `CUSTOM_JWT` backed by Amazon Cognito.

> **Note:** This module introduces **JWT-based authentication**. Up to now, all agent invocations used IAM (SigV4) authentication. From this point forward, the agent will validate user identity via JWT tokens — this is what enables per-user isolation in Gateway and (in Module 6) policy enforcement.

> **Documentation:** [AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html)

---

## Understanding Identity

When you enable `CUSTOM_JWT` authentication on a Gateway, the identity flow works like this:

1. **User authenticates** with Cognito and receives a JWT (ID token)
2. **Agent Runtime** receives the JWT in the `Authorization` header
3. **Agent code** forwards the JWT to Gateway via the MCP client
4. **Gateway validates** the JWT against the Cognito OIDC discovery endpoint
5. **Gateway calls** the target API using its own IAM role credentials

This means the target API receives requests authenticated by IAM (from the Gateway role), while the Gateway itself knows **who** the user is from the JWT. The agent code also decodes the JWT to extract the `sub` claim as an `actor_id` for Memory namespace isolation.

In **Module 6** we will add Cedar policies to Gateway that use this JWT identity to enforce per-user access control.

> **Documentation:** [AgentCore Identity](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/identity.html)

---

## Gather prerequisites

We need several values from the CloudFormation prerequisites stack: the Gateway IAM role, the REST API ID for the Task API, and Cognito details for JWT authentication.

In [ ]:
import sys; sys.path.insert(0, '..')
import boto3
from shared import utils

region = utils.get_region()
cfn = utils.get_all_cfn_outputs()

# Values from the CloudFormation prerequisites stack
gateway_role_arn = cfn.get("GatewayRoleArn") or cfn.get("GatewayServiceRoleArn")
rest_api_id = cfn.get("ApiGatewayRestApiId") or cfn.get("TaskApiRestApiId")
user_pool_id = cfn.get("UserPoolId") or cfn.get("CognitoUserPoolId")
cognito_client_id = cfn.get("UserPoolClientId") or cfn.get("CognitoClientId")

oidc_discovery_url = (
    f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}"
    f"/.well-known/openid-configuration"
) if user_pool_id else None

print(f"Region:            {region}")
print(f"Gateway Role ARN:  {gateway_role_arn}")
print(f"REST API ID:       {rest_api_id}")
print(f"User Pool ID:      {user_pool_id}")
print(f"Cognito Client ID: {cognito_client_id}")
print(f"OIDC Discovery:    {oidc_discovery_url}")

---

## Create the Gateway

The `create_gateway` API sets up a managed MCP endpoint. We configure:
- **`protocolType: MCP`** -- the gateway speaks the Model Context Protocol
- **`authorizerType: CUSTOM_JWT`** -- every request must carry a valid JWT
- **`discoveryUrl`** -- points to the Cognito OIDC well-known endpoint so Gateway can validate tokens
- **`allowedAudience`** -- restricts which Cognito app client IDs are accepted

In [ ]:
from botocore.exceptions import ClientError

control = boto3.client("bedrock-agentcore-control", region_name=region)

try:
    resp = control.create_gateway(
        name="aria-gateway",
        description="AgentCore Gateway for Aria — routes MCP tool requests to backend APIs",
        roleArn=gateway_role_arn,
        protocolType="MCP",
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": oidc_discovery_url,
                "allowedAudience": [cognito_client_id],
            }
        },
    )
    gateway_id = resp["gatewayId"]
    print(f"Gateway created: {gateway_id}")
    print(f"   URL: {resp.get('gatewayUrl', '')}")
    print(f"   Status: {resp['status']}")

except ClientError as e:
    if e.response["Error"]["Code"] in ("ConflictException", "ValidationException"):
        print("Gateway already exists -- looking it up...")
        paginator = control.get_paginator("list_gateways")
        for page in paginator.paginate():
            for gw in page.get("items", []):
                if gw["name"] == "aria-gateway":
                    gateway_id = gw["gatewayId"]
                    detail = control.get_gateway(gatewayIdentifier=gateway_id)
                    print(f"Found: {gateway_id}")
                    print(f"   URL: {detail.get('gatewayUrl', '')}")
                    break
    else:
        raise

### Wait for Gateway to become READY

Gateway creation is asynchronous. We poll until the status reaches `READY` and capture the gateway URL.

In [ ]:
import time

for i in range(30):
    gw = control.get_gateway(gatewayIdentifier=gateway_id)
    status = gw["status"]
    print(f"  [{i*5}s] Status: {status}")
    if status == "READY":
        gateway_url = gw["gatewayUrl"]
        gateway_arn = gw.get("gatewayArn", "")
        print(f"\nGateway ready!")
        print(f"   URL: {gateway_url}")
        print(f"   ARN: {gateway_arn}")
        break
    if status in ("CREATE_FAILED", "FAILED"):
        print(f"Failed: {gw.get('statusReasons', '')}")
        break
    time.sleep(5)

---

## Add the Task API target

A Gateway **target** connects the MCP endpoint to a backend API. We point it at the API Gateway REST API from the prerequisites stack.

Key configuration:
- **`apiGateway`** — tells Gateway to **auto-discover** the REST API structure from the given `restApiId` and `stage`. Gateway inspects the API's resources, methods, and models to generate MCP tools automatically.
- **`toolOverrides`** — customizes the auto-discovered tools with clearer names and descriptions so the agent understands what each tool does. Without overrides, tool names are generated from HTTP method + path (e.g., `get_tasks`).
- **`toolFilters`** — restricts which API paths/methods are exposed as tools (defense in depth)
- **`GATEWAY_IAM_ROLE`** — outbound calls to the target API use the Gateway's IAM role

In [ ]:
try:
    target_resp = control.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name="TaskApi",
        description="Task Management REST API — CRUD operations for user tasks",
        targetConfiguration={
            "mcp": {
                "apiGateway": {
                    "restApiId": rest_api_id,
                    "stage": "prod",
                    "apiGatewayToolConfiguration": {
                        "toolOverrides": [
                            {"path": "/tasks", "method": "GET", "name": "list_tasks",
                             "description": "List all tasks for the current user"},
                            {"path": "/tasks", "method": "POST", "name": "create_task",
                             "description": "Create a new task. Requires 'title' in JSON body."},
                            {"path": "/tasks/{id}", "method": "PUT", "name": "update_task",
                             "description": "Update an existing task by ID."},
                            {"path": "/tasks/{id}", "method": "DELETE", "name": "delete_task",
                             "description": "Delete a task by ID."},
                        ],
                        "toolFilters": [
                            {"filterPath": "/tasks", "methods": ["GET", "POST"]},
                            {"filterPath": "/tasks/{id}", "methods": ["PUT", "DELETE"]},
                        ],
                    },
                }
            }
        },
        credentialProviderConfigurations=[
            {"credentialProviderType": "GATEWAY_IAM_ROLE"}
        ],
    )
    target_id = target_resp["targetId"]
    print(f"Target created: {target_id}")

except ClientError as e:
    if e.response["Error"]["Code"] in ("ConflictException", "ValidationException"):
        print("Target already exists")
    else:
        raise

### Save Gateway config for later modules

In [ ]:
utils.save_config("gateway", {
    "gateway_id": gateway_id,
    "gateway_url": gateway_url,
    "gateway_arn": gateway_arn,
    "region": region,
})
print("Gateway config saved for later modules")

---
## Enable Tracing for Gateway

Enable **Tracing** on the Gateway resource so that MCP tool requests and policy evaluations appear in the AgentCore Observability dashboard.

1. Open the **Amazon Bedrock AgentCore** console and navigate to **Gateways**
2. Select the **aria-gateway** resource
3. Scroll down to the **Tracing** section and click **Edit**

![Tracing section](../shared/img/tracing-01.png)

4. Toggle **Enable** on and click **Save**

![Enable tracing](../shared/img/tracing-02.png)

To see the full agent code, open [agent/main.py](agent/main.py) in a new tab.

---

## Deploy with Gateway

We redeploy Aria with both the `MEMORY_ID` (from Module 4) and the new `GATEWAY_ENDPOINT` environment variables.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared.deploy_agent import deploy
from shared.utils import load_config

memory_cfg = load_config("memory")

env_vars = {}
if memory_cfg:
    env_vars["MEMORY_ID"] = memory_cfg["memory_id"]
env_vars["GATEWAY_ENDPOINT"] = gateway_url

result = deploy(
    agent_dir="agent",
    env_vars=env_vars,
)
runtime_arn = result["runtime_arn"]

---

## Test Task Management

Now let's use Aria's new task management capabilities. Since Gateway uses `CUSTOM_JWT` auth, we first get a JWT token, then use the shared `test_agent` helper to invoke Aria.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared import test_agent

jwt_token = test_agent.get_test_token()

# Create a task
result = test_agent.invoke(
    "Create a task: Learn about AgentCore Gateway",
    jwt_token=jwt_token,
)

### List all tasks

In [ ]:
# List tasks — same session so Aria has conversation context
result = test_agent.invoke(
    "List all my tasks",
    session_id=result["session_id"],
    jwt_token=jwt_token,
)

### Mark a task as completed

In [ ]:
# Complete a task
result = test_agent.invoke(
    "Mark the AgentCore Gateway task as completed",
    session_id=result["session_id"],
    jwt_token=jwt_token,
)


### Try it interactively

You can also test Aria with the CLI chat client. Since we now have JWT authentication, use the `--auth` flag:

```bash
cd /workshop/05-gateway-identity
python ../shared/chat.py --auth
```

Try creating, listing, and completing tasks through natural conversation.

> **Documentation:** [Authenticate and authorize with Inbound Auth and Outbound Auth](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/identity.html)

---

## What's next

Gateway connects Aria to external APIs with full identity propagation. But right now, **any authenticated user can do anything** -- create tasks, delete tasks, access any endpoint.

In **Module 6: Policy**, we will add **Cedar policies** to Gateway that control exactly which actions each user is permitted to perform.

---

## Progress

In [ ]:
import sys; sys.path.insert(0, '..')
from shared.progress import show

show("05")

---

**Next up: [Module 6 -- Enforce Policies with Cedar](../06-policy/notebook.ipynb)**